# Lab 03 — Lemma/POS baseline (UA)

Трек A (класифікація). Дані беремо з Lab2: `../project_lab2/data/processed_v2/processed_v2.csv`.
Порівнюємо baseline на `processed_v2` vs `lemma_text` (і опційно додаємо POS n-grams).
В кінці: error analysis + генерація `docs/audit_summary_lab3.md`.

## 1) Install deps (Colab / local)

In [4]:
!pip -q install -r ../requirements.txt

## 2) Load processed_v2 from Lab2

In [5]:
from pathlib import Path
import pandas as pd

LAB3_ROOT = Path('..').resolve()
LAB2_ROOT = (LAB3_ROOT.parent / 'project_lab2').resolve()
v2_path = LAB2_ROOT / 'data' / 'processed_v2' / 'processed_v2.csv'
print('Reading:', v2_path)
df = pd.read_csv(v2_path)
print(df.shape)
df.head()

Reading: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\data\processed_v2\processed_v2.csv
(1000, 4)


,text_id,text,sentences,label
0,9905,"Вступив на ІСТ цього року, тепер молюся, щоб п...","[""Вступив на ІСТ цього року, тепер молюся, щоб...",Question / Request for Help
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,"[""Цифрова держава Повідомлення 123 від 18.04.2...",Question / Request for Help
2,3099,Старий університет поки що вчить. Наразі налаш...,"[""Старий університет поки що вчить."", ""Наразі ...",Neutral Comment
3,8664,"На пл. Ринок ЦНАП м.Львова, швидке ьа якісне в...","[""На пл."", ""Ринок ЦНАП м.Львова, швидке ьа які...",Gratitude / Positive Feedback
4,1035,"Мені здається, що наша кузня супер-кадрів в IT...","[""Мені здається, що наша кузня супер-кадрів в ...",Suggestion / Idea


## 3) Lemma/POS extraction with Stanza

In [6]:
import sys
from pathlib import Path
LAB3_ROOT = Path("..").resolve()
sys.path.insert(0, str(LAB3_ROOT))

from tqdm import tqdm
from src.ling_features import build_stanza_pipeline, add_ling_features

nlp = build_stanza_pipeline()

lemma_texts = []
upos_seqs = []

for t in tqdm(df['text'].astype(str).tolist()):
    ling = add_ling_features(t, nlp)
    lemma_texts.append(ling.lemma_text)
    upos_seqs.append(ling.upos_seq)

df['lemma_text'] = lemma_texts
df['upos_seq'] = upos_seqs

df[['text_id','label','text','lemma_text','upos_seq']].head(3)

100%|██████████| 1000/1000 [05:02<00:00,  3.31it/s]


,text_id,label,text,lemma_text,upos_seq
0,9905,Question / Request for Help,"Вступив на ІСТ цього року, тепер молюся, щоб п...","вступити на ІСТ цей рік , тепер молітися , щоб...",VERB ADP NOUN DET NOUN PUNCT ADV VERB PUNCT SC...
1,10001201,Question / Request for Help,Цифрова держава Повідомлення 123 від 18.04.202...,цифровий держава повідомлення 123 від 18 . 04 ...,ADJ NOUN NOUN NUM ADP ADJ PUNCT ADJ PUNCT ADJ ...
2,3099,Neutral Comment,Старий університет поки що вчить. Наразі налаш...,старий університет поки що вчити . наразі нала...,ADJ NOUN ADV PRON VERB PUNCT ADV ADJ ADP NOUN ...


## 4) Baseline comparison (TF-IDF + Linear SVM)

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

y = df['label'].astype(str)

def eval_model(name, X_text):
    X_train, X_test, y_train, y_test = train_test_split(
        X_text, y, test_size=0.2, random_state=42, stratify=y
    )
    clf = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2)),
        ('svm', LinearSVC()),
    ])
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    mf1 = f1_score(y_test, pred, average='macro')
    print(f"{name}: acc={acc:.4f}, macroF1={mf1:.4f}")
    return acc, mf1, classification_report(y_test, pred, digits=4)

acc1, mf1_1, rep1 = eval_model('processed_v2', df['text'].astype(str))
acc2, mf1_2, rep2 = eval_model('lemma_text', df['lemma_text'].astype(str))

processed_v2: acc=0.6250, macroF1=0.6260
lemma_text: acc=0.7050, macroF1=0.7031


## 5) Optional: add POS n-grams as extra features

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

Xdf = df[['text','upos_seq']].copy()
X_train, X_test, y_train, y_test = train_test_split(
    Xdf, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ('features', ColumnTransformer([
        ('text_tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2), 'text'),
        ('pos_tfidf', TfidfVectorizer(ngram_range=(2,4), min_df=2), 'upos_seq'),
    ])),
    ('clf', LogisticRegression(max_iter=2000)),
])

model.fit(X_train, y_train)
pred = model.predict(X_test)
acc3 = accuracy_score(y_test, pred)
mf1_3 = f1_score(y_test, pred, average='macro')
print(f"text+POS: acc={acc3:.4f}, macroF1={mf1_3:.4f}")

text+POS: acc=0.5950, macroF1=0.5958


## 6) Error analysis: show 10 real lemma/POS issues

In [9]:
import re

def looks_tricky(t: str) -> bool:
    t = str(t)
    return bool(
        re.search(r"[A-Za-z]", t) or
        re.search(r"\d+\.\d+|\d+\.\d+\.\d+", t) or
        ("'" in t) or ("ʼ" in t) or ("’" in t) or
        ("-" in t) or ("—" in t) or ("–" in t) or
        ("@" in t) or ("#" in t)
    )

tricky = df[df['text'].apply(looks_tricky)]
tricky = tricky.sample(10, random_state=42) if len(tricky) >= 10 else tricky
tricky[['text_id','label','text','lemma_text','upos_seq']]

,text_id,label,text,lemma_text,upos_seq
234,11919,Gratitude / Positive Feedback,Дуже задоволений спортивною базою коледжу! Тре...,дуже задоволений спортивний база коледж ! трен...,ADV ADJ ADJ NOUN NOUN PUNCT NOUN PUNCT ADJ NOU...
884,3924,Neutral Comment,На площі Конституції знаходиться адміністратив...,на площа конституція знаходитися адміністратив...,ADP NOUN NOUN VERB ADJ NOUN ADJ NOUN ADJ NOUN ...
63,13831,Question / Request for Help,"Підкажіть будь ласка контактні дані, по яким м...","підказати бути ласка контактний дані , по який...",VERB VERB NOUN ADJ NOUN PUNCT ADP DET ADV VERB...
826,8802,Complaint / Dissatisfaction,"Жахливо!!! Мій чоловік іноземець , завдяки цим...","жахливо !!! мій чоловік іноземець , завдяки це...",ADV PUNCT DET NOUN NOUN PUNCT ADP DET PUNCT NO...
20,15243,Gratitude / Positive Feedback,"Мій ФОП, зареєстрований у Львові, але я живу в...","мій ФОП , зареєстрований у Львів , але я жити ...",DET NOUN PUNCT ADJ ADP PROPN PUNCT CCONJ PRON ...
322,4248,Complaint / Dissatisfaction,Чудовий університет. Дуже зручне розташування....,чудовий університет . дуже зручний розташуванн...,ADJ NOUN PUNCT ADV ADJ NOUN PUNCT ADP NOUN PUN...
557,4574,Neutral Comment,Колишнє відділення Харківського земельного бан...,колишній відділення харківський земельний банк...,ADJ NOUN ADJ ADJ NOUN PUNCT DET VERB NOUN ADP ...
594,16669,Suggestion / Idea,Найвидатніша архітектурна пам'ятка міста обов'...,найвидатніший архітектурний пам’ятка місто обо...,ADJ ADJ NOUN NOUN ADV VERB PUNCT SCONJ AUX ADP...
193,10001643,Question / Request for Help,Це інформація з відео у мережах чи ви самі опл...,це інформація з відео у мережа чи ви сам оплач...,PRON NOUN ADP NOUN ADP NOUN CCONJ PRON DET VER...
371,16625,Suggestion / Idea,"Якщо відвідуєте Чернівці, обов'язково заплануй...","якщо відвідувати Чернівка , обов’язково заплан...",SCONJ VERB PROPN PUNCT ADV VERB NOUN NUM ADP D...


In [10]:
from src.ling_features import add_ling_features

for _, r in tricky.iterrows():
    ling = add_ling_features(r['text'], nlp)
    print('\n---')
    print('text_id:', r['text_id'], '| label:', r['label'])
    print('processed_v2:', r['text'])
    print('TOK:', ling.tokens[:30])
    print('LEM:', ling.lemmas[:30])
    print('POS:', ling.upos[:30])
    print('comment: (дописати) що пішло не так і чи критично для класифікації')



---
text_id: 11919 | label: Gratitude / Positive Feedback
processed_v2: Дуже задоволений спортивною базою коледжу! Тренери - справжні професіонали, завжди підтримують і мотивують. Завдяки їм маємо чудові результати на змаганнях. Атмосфера дружня, заняття цікаві й ефективні. Коледж дбає про всебічний розвиток студентів - це великий плюс!
TOK: ['Дуже', 'задоволений', 'спортивною', 'базою', 'коледжу', '!', 'Тренери', '-', 'справжні', 'професіонали', ',', 'завжди', 'підтримують', 'і', 'мотивують', '.', 'Завдяки', 'їм', 'маємо', 'чудові', 'результати', 'на', 'змаганнях', '.', 'Атмосфера', 'дружня', ',', 'заняття', 'цікаві', 'й']
LEM: ['дуже', 'задоволений', 'спортивний', 'база', 'коледж', '!', 'тренер', '-', 'справжній', 'професіонал', ',', 'завжди', 'підтримувати', 'і', 'мотивувати', '.', 'завдяки', 'вони', 'мати', 'чудовий', 'результат', 'на', 'змагання', '.', 'атмосфера', 'дружній', ',', 'заняття', 'цікавий', 'й']
POS: ['ADV', 'ADJ', 'ADJ', 'NOUN', 'NOUN', 'PUNCT', 'NOUN', 'PUNCT', 'ADJ

## 7) Generate ling_edge_cases.jsonl (10–20 cases)

In [11]:
import json
from pathlib import Path

out_path = LAB3_ROOT / 'tests' / 'ling_edge_cases.jsonl'

def score(t: str) -> int:
    t = str(t)
    s = 0
    if re.search(r"[A-Za-z]", t): s += 2
    if "'" in t or "’" in t or "ʼ" in t: s += 2
    if "-" in t or "—" in t or "–" in t: s += 1
    if re.search(r"[@#]", t): s += 1
    if re.search(r"\d+\.\d+|\d+\.\d+\.\d+", t): s += 2
    if t.isupper() and len(t) >= 6: s += 2
    return s

cand = df.sample(min(len(df), 400), random_state=42).copy()
cand['score'] = cand['text'].apply(score)
cand = cand.sort_values('score', ascending=False).head(80)
chosen = cand.sample(min(20, len(cand)), random_state=42)

rows = []
for _, r in chosen.iterrows():
    ling = add_ling_features(r['text'], nlp)
    rows.append({
        'text_id': int(r['text_id']),
        'label': r.get('label',''),
        'processed_v2': r['text'],
        'tokens': ling.tokens[:40],
        'lemmas': ling.lemmas[:40],
        'upos': ling.upos[:40],
        'expected_behavior': 'Inspect lemma/POS for slang/translit/abbr/hyphen/apostrophe and note impact',
    })

out_path.parent.mkdir(exist_ok=True)
with out_path.open('w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print('Saved:', out_path)

Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab3\tests\ling_edge_cases.jsonl


## 8) Generate docs/audit_summary_lab3.md

In [12]:
from pathlib import Path

doc = LAB3_ROOT / 'docs' / 'audit_summary_lab3.md'
lines = []
lines.append('# Audit summary — Lab3 (Lemma/POS baseline)\n')
lines.append('## Metrics\n')
lines.append(f'- processed_v2: acc={acc1:.4f}, macroF1={mf1_1:.4f}')
lines.append(f'- lemma_text: acc={acc2:.4f}, macroF1={mf1_2:.4f}')
lines.append(f'- text+POS: acc={acc3:.4f}, macroF1={mf1_3:.4f}' if 'acc3' in globals() else '- text+POS: not run')
lines.append('\n## Decision (5–7 sentences)\n')
lines.append('Напишіть висновок: коли lemma_text покращив baseline і на скільки; де стало гірше/не змінилось; чи беремо леми/POS і як саме.')
doc.write_text('\n'.join(lines), encoding='utf-8')
print('Saved:', doc)

Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab3\docs\audit_summary_lab3.md
